In [ ]:
from pathlib import Path

from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from IPython.core.interactiveshell import InteractiveShell
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

InteractiveShell.ast_node_interactivity = "all"

In [ ]:
DATASETS = next(
    path / "datasets" for path in Path.cwd().parents if (path / "datasets").is_dir()
)
SOURCE = DATASETS / "movies_imdb"
TARGET = DATASETS / "movies_imdb_delta"

spark = configure_spark_with_delta_pip(
    SparkSession.builder.appName("XGBoost Study")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .config("spark.driver.memory", "6g")
    .config(
        "spark.sql.parquet.compression.codec", "zstd"
    )  # spark by default use snappy. but zstd can be more eficente in disc and I/O, but less in cpu optimization
    .config("spark.databricks.delta.optimizeWrite.enabled", "true")
    .config("spark.databricks.delta.autoCompact.enabled", "true")
    .config(
        "spark.databricks.delta.optimize.maxFileSize", "268435456"
    )  # optimize parquet size
    .config(
        "spark.ui.showConsoleProgress", "false"
    )  # show logs. only in developent stage
).getOrCreate()

In [ ]:
for table in sorted(SOURCE.glob("*.parquet")):
    destino = TARGET / table.stem
    if DeltaTable.isDeltaTable(spark, str(destino)):
        continue
    data = spark.read.parquet(str(table))
    data.write.format("delta").mode("overwrite").save(str(destino))

In [ ]:
df_basics = spark.read.format("delta").load(str(TARGET / "title.basics"))
df_basics = df_basics.drop("originalTitle", "endYear")
df_basics = df_basics.withColumnsRenamed(
    {
        "tconst": "id",
        "titleType": "type",
        "isAdult": "is_adult",
        "startYear": "year",
        "runtimeMinutes": "time_min",
        "genres": "gens",
    }
)
df_basics = df_basics.filter(F.col("type") == "movie")
df_basics = df_basics.withColumn(
    "gens",
    F.when(
        (F.col("gens").isNull()) | (F.col("gens") == "") | (F.col("gens") == r"\N"),
        F.array(F.lit("non-categorized")),
    ).otherwise(F.split(F.col("gens"), ",")),
)
df_basics.printSchema()
df_basics.show(10, truncate=False)
df_basics.count()
# df_basics.summary().show()

In [ ]:
df_principals = spark.read.format("delta").load(str(TARGET / "title.principals"))
df_principals.printSchema()
df_principals.show(10, truncate=False)

In [ ]:
df_name_basics = spark.read.format("delta").load(str(TARGET / "name.basics"))
df_name_basics.printSchema()
df_name_basics.show(10, truncate=False)

In [ ]:
df_akas = spark.read.format("delta").load(str(TARGET / "title.akas"))
df_akas.printSchema()
df_akas.show(10, truncate=False)

In [ ]:
df_crew = spark.read.format("delta").load(str(TARGET / "title.crew"))
df_crew.printSchema()
df_crew.show(10, truncate=False)

In [ ]:
df_episode = spark.read.format("delta").load(str(TARGET / "title.episode"))
df_episode.printSchema()
df_episode.show(10, truncate=False)

In [ ]:
df_ratings = spark.read.format("delta").load(str(TARGET / "title.ratings"))
df_ratings.printSchema()
df_ratings.show(10, truncate=False)